# Oral Cancer Risk Prediction Using Machine Learning
## Exploratory Data Analysis & Model Training Pipeline

This notebook demonstrates the end-to-end pipeline for the academic project, including data acquisition, cleaning, preprocessing, exploratory data analysis, model training, comparison, and selection.

### 1. Environment Setup & Data Preprocessing
We import our custom modules and execute the data preprocessing pipeline. This will:
1. Programmatically acquire the public patient-level dataset.
2. Filter out data leakage columns (e.g. tumor size, cancer stage, cost, survival rates).
3. Apply stratified splitting (80/20 train/test).
4. Construct and save the fitted preprocessing pipeline (`preprocessing.pkl`).

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Ensure project root is in the path
sys.path.append(os.path.abspath('..'))

from src.data_preprocessing import preprocess_and_split

# Run data download and preprocessing split
preprocess_and_split()

### 2. Exploratory Data Analysis (EDA)
Let's load the raw dataset and inspect the characteristics of our features and the target distribution.

In [ ]:
# Load raw dataset
raw_data_path = os.path.join('..', 'data', 'raw', 'oral_cancer_prediction_dataset.csv')
df = pd.read_csv(raw_data_path)

print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Class distribution of target variable
target_counts = df['Oral Cancer (Diagnosis)'].value_counts()
target_pct = df['Oral Cancer (Diagnosis)'].value_counts(normalize=True) * 100

plt.figure(figsize=(6, 4))
sns.barplot(x=target_counts.index, y=target_counts.values, palette='Blues_r')
plt.title('Target Variable Distribution (Oral Cancer)')
plt.ylabel('Number of Patients')
plt.xlabel('Oral Cancer Diagnosis')
for i, val in enumerate(target_counts.values):
    plt.text(i, val + 1000, f"{val} ({target_pct.iloc[i]:.1f}%)", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Age Distribution by Cancer Outcome
plt.figure(figsize=(9, 5))
sns.histplot(data=df, x='Age', hue='Oral Cancer (Diagnosis)', multiple='stack', kde=True, bins=30, palette='Set2')
plt.title('Age Distribution by Oral Cancer Diagnosis Status')
plt.xlabel('Age (Years)')
plt.ylabel('Count')
plt.show()

In [ ]:
# Lifestyle Risks Analysis: Tobacco and Alcohol correlation
cross_tab = pd.crosstab(df['Tobacco Use'], df['Oral Cancer (Diagnosis)'], normalize='index') * 100
print("Percentage of patients with Oral Cancer by Tobacco Use Status:")
print(cross_tab)

plt.figure(figsize=(6, 4))
cross_tab.plot(kind='bar', stacked=True, color=['#7fcdbb', '#fc8d59'])
plt.title('Oral Cancer Rates by Tobacco Use')
plt.ylabel('Percentage (%)')
plt.xlabel('Tobacco Use')
plt.legend(title='Oral Cancer')
plt.tight_layout()
plt.show()

### 3. Model Training & Comparison
Now we execute our training pipeline. It trains:
- Logistic Regression (Balanced)
- Decision Tree (Balanced)
- Random Forest (Balanced)
- XGBoost (Weighted)

It uses 5-fold Stratified Cross-Validation on the training split, evaluates them on the held-out test split, and writes comparison logs.

In [ ]:
from src.train import train_and_evaluate

# Run training and evaluation pipeline
train_and_evaluate()

In [ ]:
# Load and display model comparison results
comparison_path = os.path.join('..', 'reports', 'model_comparison.csv')
comparison_df = pd.read_csv(comparison_path)

print("\n--- Model Performance Comparison (Sorted by Test F1-Score) ---")
comparison_df.sort_values(by='Test F1-Score', ascending=False)

### 4. Evaluation Plots of the Selected Best Model
Let's generate the figures (Confusion Matrix, ROC Curve, and Feature Importance) for our best selected model.

In [ ]:
from src.evaluate import generate_evaluation_plots

# Generate plots
generate_evaluation_plots()

In [ ]:
# Display generated Confusion Matrix and ROC Curve side-by-side
from IPython.display import Image, display

cm_fig = os.path.join('..', 'reports', 'figures', 'confusion_matrix.png')
roc_fig = os.path.join('..', 'reports', 'figures', 'roc_curve.png')
feat_fig = os.path.join('..', 'reports', 'figures', 'feature_importance.png')

if os.path.exists(cm_fig):
    print("\nConfusion Matrix:")
    display(Image(filename=cm_fig))
if os.path.exists(roc_fig):
    print("\nROC Curve:")
    display(Image(filename=roc_fig))
if os.path.exists(feat_fig):
    print("\nFeature Importance / Coefficients:")
    display(Image(filename=feat_fig))